In [14]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error

import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping

print("TensorFlow:", tf.__version__)

TensorFlow: 2.21.0


DATA LOADING..

In [12]:
file_path = "/home/techpark-2/Desktop/radiation/dataset/proton_1_500MeV_clean.csv"

df = pd.read_csv(file_path)

df["time"] = pd.to_datetime(df["time"])

df = df.sort_values("time")
print("Rows:", len(df))
print("Columns:", len(df.columns))

df.head()

Rows: 1736640
Columns: 14


,time,1.02_1.86_MeV,1.9_2.3_MeV,2.31_3.34_MeV,3.4_6.48_MeV,5.84_11_MeV,11.64_23.27_MeV,23.9_32.6_MeV,40.7_68.2_MeV,83.9_98.4_MeV,99.7_118_MeV,123_148_MeV,156_237_MeV,267_390_MeV
0,2022-09-13 00:00:00,0.035267,0.009745,0.003481,0.000167,0.000181,0.000123,0.000021,0.000005,0.000000e+00,0.000001,0.000003,1.385365e-06,3.103786e-07
1,2022-09-13 00:01:00,0.036756,0.012085,0.003330,0.000056,0.000000,0.000098,0.000038,0.000001,0.000000e+00,0.000000,0.000003,1.385347e-06,1.034578e-07
2,2022-09-13 00:02:00,0.032474,0.007404,0.003481,0.000111,0.000060,0.000160,0.000038,0.000004,0.000000e+00,0.000000,0.000003,1.847152e-06,2.069206e-07
3,2022-09-13 00:03:00,0.033030,0.012080,0.004086,0.000167,0.000242,0.000049,0.000048,0.000005,5.671780e-07,0.000002,0.000004,4.617816e-07,4.138364e-07
4,2022-09-13 00:04:00,0.033216,0.013639,0.003329,0.000111,0.000030,0.000086,0.000024,0.000011,5.671729e-07,0.000002,0.000002,6.926708e-07,3.103777e-07


In [9]:
df

,time,1.02_1.86_MeV,1.9_2.3_MeV,2.31_3.34_MeV,3.4_6.48_MeV,5.84_11_MeV,11.64_23.27_MeV,23.9_32.6_MeV,40.7_68.2_MeV,83.9_98.4_MeV,99.7_118_MeV,123_148_MeV,156_237_MeV,267_390_MeV
0,2022-09-13 00:00:00,0.035267,0.009745,0.003481,0.000167,0.000181,0.000123,0.000021,0.000005,0.000000e+00,0.000001,0.000003,1.385365e-06,3.103786e-07
1,2022-09-13 00:01:00,0.036756,0.012085,0.003330,0.000056,0.000000,0.000098,0.000038,0.000001,0.000000e+00,0.000000,0.000003,1.385347e-06,1.034578e-07
2,2022-09-13 00:02:00,0.032474,0.007404,0.003481,0.000111,0.000060,0.000160,0.000038,0.000004,0.000000e+00,0.000000,0.000003,1.847152e-06,2.069206e-07
3,2022-09-13 00:03:00,0.033030,0.012080,0.004086,0.000167,0.000242,0.000049,0.000048,0.000005,5.671780e-07,0.000002,0.000004,4.617816e-07,4.138364e-07
4,2022-09-13 00:04:00,0.033216,0.013639,0.003329,0.000111,0.000030,0.000086,0.000024,0.000011,5.671729e-07,0.000002,0.000002,6.926708e-07,3.103777e-07
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1736635,2025-12-31 23:55:00,0.003991,0.000000,0.000000,0.000000,0.000059,0.000012,0.000000,0.000001,5.670794e-07,0.000001,0.000001,6.925553e-07,0.000000e+00
1736636,2025-12-31 23:56:00,0.005986,0.000000,0.000000,0.000000,0.000000,0.000024,0.000000,0.000001,5.670794e-07,0.000001,0.000001,6.925553e-07,0.000000e+00
1736637,2025-12-31 23:57:00,0.005986,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000001,5.670794e-07,0.000001,0.000001,6.925553e-07,1.034416e-07
1736638,2025-12-31 23:58:00,0.004716,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000001,5.670794e-07,0.000001,0.000001,6.925553e-07,2.068815e-07


In [15]:
SETTING TIME INDEX

SyntaxError: invalid syntax (1887599928.py, line 1)

In [16]:
df = df.set_index("time")

df.head()

,1.02_1.86_MeV,1.9_2.3_MeV,2.31_3.34_MeV,3.4_6.48_MeV,5.84_11_MeV,11.64_23.27_MeV,23.9_32.6_MeV,40.7_68.2_MeV,83.9_98.4_MeV,99.7_118_MeV,123_148_MeV,156_237_MeV,267_390_MeV
time,,,,,,,,,,,,,
2022-09-13 00:00:00,0.035267,0.009745,0.003481,0.000167,0.000181,0.000123,0.000021,0.000005,0.000000e+00,0.000001,0.000003,1.385365e-06,3.103786e-07
2022-09-13 00:01:00,0.036756,0.012085,0.003330,0.000056,0.000000,0.000098,0.000038,0.000001,0.000000e+00,0.000000,0.000003,1.385347e-06,1.034578e-07
2022-09-13 00:02:00,0.032474,0.007404,0.003481,0.000111,0.000060,0.000160,0.000038,0.000004,0.000000e+00,0.000000,0.000003,1.847152e-06,2.069206e-07
2022-09-13 00:03:00,0.033030,0.012080,0.004086,0.000167,0.000242,0.000049,0.000048,0.000005,5.671780e-07,0.000002,0.000004,4.617816e-07,4.138364e-07
2022-09-13 00:04:00,0.033216,0.013639,0.003329,0.000111,0.000030,0.000086,0.000024,0.000011,5.671729e-07,0.000002,0.000002,6.926708e-07,3.103777e-07


SETTING INTERVALS

In [17]:

intervals = df.index.to_series().diff().iloc[1:]

bad_intervals = intervals[
    intervals != pd.Timedelta(minutes=1)
]

print("Total rows:", len(df))
print("Incorrect intervals:", len(bad_intervals))

bad_intervals.head()

Total rows: 1736640
Incorrect intervals: 0


Series([], Name: time, dtype: timedelta64[us])

13 FEATURES

In [18]:
features = df.columns.tolist()

print("Number of features:", len(features))

for i, feature in enumerate(features):
    print(i, feature)

Number of features: 13
0 1.02_1.86_MeV
1 1.9_2.3_MeV
2 2.31_3.34_MeV
3 3.4_6.48_MeV
4 5.84_11_MeV
5 11.64_23.27_MeV
6 23.9_32.6_MeV
7 40.7_68.2_MeV
8 83.9_98.4_MeV
9 99.7_118_MeV
10 123_148_MeV
11 156_237_MeV
12 267_390_MeV


FINDING MISSING DATAS

In [19]:
df[features].isna().sum()

1.02_1.86_MeV      0
1.9_2.3_MeV        0
2.31_3.34_MeV      0
3.4_6.48_MeV       0
5.84_11_MeV        0
11.64_23.27_MeV    0
23.9_32.6_MeV      0
40.7_68.2_MeV      0
83.9_98.4_MeV      0
99.7_118_MeV       0
123_148_MeV        0
156_237_MeV        0
267_390_MeV        0
dtype: int64

In [20]:
np.isinf(df[features].values).sum()

np.int64(0)

In [21]:
print(df)

                     1.02_1.86_MeV  1.9_2.3_MeV  2.31_3.34_MeV  3.4_6.48_MeV  \
time                                                                           
2022-09-13 00:00:00       0.035267     0.009745       0.003481      0.000167   
2022-09-13 00:01:00       0.036756     0.012085       0.003330      0.000056   
2022-09-13 00:02:00       0.032474     0.007404       0.003481      0.000111   
2022-09-13 00:03:00       0.033030     0.012080       0.004086      0.000167   
2022-09-13 00:04:00       0.033216     0.013639       0.003329      0.000111   
...                            ...          ...            ...           ...   
2025-12-31 23:55:00       0.003991     0.000000       0.000000      0.000000   
2025-12-31 23:56:00       0.005986     0.000000       0.000000      0.000000   
2025-12-31 23:57:00       0.005986     0.000000       0.000000      0.000000   
2025-12-31 23:58:00       0.004716     0.000000       0.000000      0.000000   
2025-12-31 23:59:00       0.005079     0

DATA EXTRACTION

In [22]:
data = df[features].values.astype(np.float32)

print(data.shape)

(1736640, 13)


MODEL SPLIT

In [23]:
n = len(data)
train_end = int(n * 0.70)
val_end = int(n * 0.85)

train_data = data[:train_end]
val_data = data[train_end:val_end]
test_data = data[val_end:]

print("Train:", train_data.shape)
print("Validation:", val_data.shape)
print("Test:", test_data.shape)

Train: (1215648, 13)
Validation: (260496, 13)
Test: (260496, 13)


In [24]:
EPSILON = 1e-12

train_log = np.log10(train_data + EPSILON)
val_log = np.log10(val_data + EPSILON)
test_log = np.log10(test_data + EPSILON)

SCALING

In [25]:
scaler = MinMaxScaler()

train_scaled = scaler.fit_transform(train_log)

val_scaled = scaler.transform(val_log)
test_scaled = scaler.transform(test_log)

print(train_scaled.min())
print(train_scaled.max())

0.0
1.0000001


In [26]:
LOOKBACK = 60

In [27]:
def create_sequences(data, lookback):
    X = []
    y = []

    for i in range(lookback, len(data)):
        X.append(data[i-lookback:i])
        y.append(data[i])

    return np.array(X), np.array(y)

In [ ]:
X_train, y_train = create_sequences(train_scaled, LOOKBACK)

X_val, y_val = create_sequences(val_scaled, LOOKBACK)

X_test, y_test = create_sequences(test_scaled, LOOKBACK)

In [29]:
print("X_train:", X_train.shape)
print("y_train:", y_train.shape)

print("X_val:", X_val.shape)
print("y_val:", y_val.shape)

print("X_test:", X_test.shape)
print("y_test:", y_test.shape)

X_train: (1215588, 60, 13)
y_train: (1215588, 13)
X_val: (260436, 60, 13)
y_val: (260436, 13)
X_test: (260436, 60, 13)
y_test: (260436, 13)


BUILD THE LSTM

In [38]:
import os
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dropout, Dense
from tensorflow.keras.callbacks import EarlyStopping
import tensorflow as tf
LOOKBACK = 60
print(tf.config.list_physical_devices('GPU'))
os.makedirs("saved_model", exist_ok=True)

model = Sequential([
    LSTM(
        64,
        input_shape=(LOOKBACK, len(features))
    ),

    Dropout(0.1),

    Dense(32, activation="relu"),

    Dense(len(features))
])

model.compile(
    optimizer="adam",
    loss="mse",
    metrics=["mae"]
)

early_stopping = EarlyStopping(
    monitor="val_loss",
    patience=5,
    restore_best_weights=True
)

history = model.fit(
    X_train,
    y_train,
    validation_data=(X_val, y_val),
    epochs=50,
    batch_size=256,
    callbacks=[early_stopping],
    verbose=1
)

model.save(f"saved_model/trained_model01.keras")


[PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]
Epoch 1/50
4749/4749 ━━━━━━━━━━━━━━━━━━━━ 18s 4ms/step - loss: 0.0424 - mae: 0.1311 - val_loss: 0.0340 - val_mae: 0.1092
Epoch 2/50
4749/4749 ━━━━━━━━━━━━━━━━━━━━ 17s 4ms/step - loss: 0.0401 - mae: 0.1213 - val_loss: 0.0340 - val_mae: 0.1080
Epoch 3/50
4749/4749 ━━━━━━━━━━━━━━━━━━━━ 17s 4ms/step - loss: 0.0399 - mae: 0.1201 - val_loss: 0.0338 - val_mae: 0.1074
Epoch 4/50
4749/4749 ━━━━━━━━━━━━━━━━━━━━ 17s 4ms/step - loss: 0.0399 - mae: 0.1194 - val_loss: 0.0339 - val_mae: 0.1070
Epoch 5/50
4749/4749 ━━━━━━━━━━━━━━━━━━━━ 18s 4ms/step - loss: 0.0398 - mae: 0.1190 - val_loss: 0.0338 - val_mae: 0.1074
Epoch 6/50
4749/4749 ━━━━━━━━━━━━━━━━━━━━ 18s 4ms/step - loss: 0.0398 - mae: 0.1186 - val_loss: 0.0338 - val_mae: 0.1075
Epoch 7/50
4749/4749 ━━━━━━━━━━━━━━━━━━━━ 18s 4ms/step - loss: 0.0397 - mae: 0.1184 - val_loss: 0.0338 - val_mae: 0.1082
Epoch 8/50
4749/4749 ━━━━━━━━━━━━━━━━━━━━ 18s 4ms/step - loss: 0.0397 - mae: 0.1182 - 

**V2 TRAINING**

In [43]:
# ============================================================
# 1. IMPORTS
# ============================================================

import os

# Allow TensorFlow to grow GPU memory as needed
os.environ["TF_GPU_ALLOCATOR"] = "cuda_malloc_async"

import numpy as np
import pandas as pd
import tensorflow as tf

from tensorflow import keras
from tensorflow.keras import Sequential
from tensorflow.keras.layers import Input, LSTM, Dense, Dropout
from tensorflow.keras.callbacks import (
    EarlyStopping,
    ReduceLROnPlateau,
    ModelCheckpoint
)

from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score,
    explained_variance_score
)

import joblib


# ============================================================
# 2. GPU CONFIGURATION
# ============================================================

print("TensorFlow:", tf.__version__)

gpus = tf.config.list_physical_devices("GPU")

print("Available GPUs:", gpus)

if not gpus:
    raise RuntimeError("GPU not detected.")

# Memory growth must be configured before GPU initialization.
try:
    for gpu in gpus:
        tf.config.experimental.set_memory_growth(gpu, True)
except RuntimeError as e:
    print("Memory growth setting skipped:", e)

print("GPU configuration complete.")


# ============================================================
# 3. RANDOM SEED
# ============================================================

SEED = 42

np.random.seed(SEED)
tf.random.set_seed(SEED)


# ============================================================
# 4. SAVE DIRECTORY
# ============================================================

SAVE_DIR = "saved_modelv3"

os.makedirs(SAVE_DIR, exist_ok=True)

print("Save directory:", SAVE_DIR)


# ============================================================
# 5. LOAD DATA
# ============================================================

CSV_FILE = "/home/techpark-2/Desktop/radiation/dataset/proton_1_500MeV_clean.csv"

df = pd.read_csv(CSV_FILE)

print("\nDataset shape:", df.shape)
print(df.head())


# ============================================================
# 6. DEFINE 13 ENERGY CHANNELS
# ============================================================

features = [
    '1.02_1.86_MeV',
    '1.9_2.3_MeV',
    '2.31_3.34_MeV',
    '3.4_6.48_MeV',
    '5.84_11_MeV',
    '11.64_23.27_MeV',
    '23.9_32.6_MeV',
    '40.7_68.2_MeV',
    '83.9_98.4_MeV',
    '99.7_118_MeV',
    '123_148_MeV',
    '156_237_MeV',
    '267_390_MeV'
]

print("\nNumber of features:", len(features))

missing = [c for c in features if c not in df.columns]

if missing:
    raise ValueError(f"Missing columns: {missing}")



# ============================================================
# 7. EXTRACT NUMERIC DATA
# ============================================================

data = df[features].to_numpy(dtype=np.float32)

print("\nData shape:", data.shape)
print("Data dtype:", data.dtype)


# ============================================================
# 8. CHECK NaN / INF
# ============================================================

print("\nNaN count:", np.isnan(data).sum())
print("Inf count:", np.isinf(data).sum())

data = np.nan_to_num(
    data,
    nan=0.0,
    posinf=0.0,
    neginf=0.0
)


# ============================================================
# 9. LOG10 TRANSFORMATION
# ============================================================

EPSILON = 1e-12

data_log = np.log10(data + EPSILON)

print("Log data shape:", data_log.shape)


# ============================================================
# 10. CHRONOLOGICAL SPLIT
#     85% TRAIN
#     10% VALIDATION
#      5% TEST
# ============================================================

n = len(data_log)

train_end = int(n * 0.85)
val_end = int(n * 0.95)

train_log = data_log[:train_end]
val_log = data_log[train_end:val_end]
test_log = data_log[val_end:]

print("\n================ DATA SPLIT ================")
print("Train:", train_log.shape)
print("Validation:", val_log.shape)
print("Test:", test_log.shape)


# ============================================================
# 11. MINMAX SCALER
#     FIT ONLY ON TRAINING DATA
# ============================================================

scaler = MinMaxScaler()

train_scaled = scaler.fit_transform(train_log)
val_scaled = scaler.transform(val_log)
test_scaled = scaler.transform(test_log)

print("\n================ SCALING ================")
print("Train:", train_scaled.shape)
print("Validation:", val_scaled.shape)
print("Test:", test_scaled.shape)


# ============================================================
# 12. SAVE SCALER
# ============================================================

scaler_path = os.path.join(
    SAVE_DIR,
    "scaler.pkl"
)

joblib.dump(
    scaler,
    scaler_path
)

print("Scaler saved:", scaler_path)


# ============================================================
# 13. CREATE TF.DATA SEQUENCES
#
#     This avoids manually creating gigantic X_train arrays
#     and lets TensorFlow feed batches efficiently.
# ============================================================

LOOKBACK =120
BATCH_SIZE = 128


def make_dataset(data, lookback, batch_size, shuffle=False):

    # Inputs:
    # data[:-1]
    #
    # Targets:
    # data[lookback:]
    #
    # Each sample:
    # previous 60 timesteps -> next timestep

    dataset = tf.keras.utils.timeseries_dataset_from_array(
        data=data[:-1],
        targets=data[lookback:],
        sequence_length=lookback,
        sequence_stride=1,
        sampling_rate=1,
        batch_size=batch_size,
        shuffle=shuffle,
        seed=SEED
    )

    # Ensure GPU receives float32 tensors
    dataset = dataset.map(
        lambda x, y: (
            tf.cast(x, tf.float32),
            tf.cast(y, tf.float32)
        ),
        num_parallel_calls=tf.data.AUTOTUNE
    )

    # Prefetch batches while GPU is working
    dataset = dataset.prefetch(
        tf.data.AUTOTUNE
    )

    return dataset


# Create datasets
train_ds = make_dataset(
    train_scaled,
    LOOKBACK,
    BATCH_SIZE,
    shuffle=True
)

val_ds = make_dataset(
    val_scaled,
    LOOKBACK,
    BATCH_SIZE,
    shuffle=False
)

test_ds = make_dataset(
    test_scaled,
    LOOKBACK,
    BATCH_SIZE,
    shuffle=False
)


print("\n================ DATASETS CREATED ================")

for x_batch, y_batch in train_ds.take(1):
    print("X batch shape:", x_batch.shape)
    print("Y batch shape:", y_batch.shape)

    break


# ============================================================
# 14. BUILD MODEL
#     SAME ARCHITECTURE AS YOUR ORIGINAL CODE
# ============================================================

model = Sequential([
    Input(
        shape=(
            LOOKBACK,
            len(features)
        )
    ),

    LSTM(
        64,
        return_sequences=True
    ),

    Dropout(0.2),

    LSTM(
        32,
        return_sequences=False
    ),

    Dropout(0.2),

    Dense(
        32,
        activation="relu"
    ),

    Dense(
        len(features),
        activation="linear"
    )
])


# ============================================================
# 15. COMPILE
# ============================================================

optimizer = keras.optimizers.Adam(
    learning_rate=1e-3
)

model.compile(
    optimizer=optimizer,
    loss="mse",
    metrics=["mae"]
)


# ============================================================
# 16. MODEL SUMMARY
# ============================================================

print("\n================ MODEL ================")

model.summary()


# ============================================================
# 17. CALLBACKS
# ============================================================

best_model_path = os.path.join(
    SAVE_DIR,
    "best_lstm_model.keras"
)

callbacks = [

    EarlyStopping(
        monitor="val_loss",
        patience=8,
        restore_best_weights=True,
        verbose=1
    ),

    ReduceLROnPlateau(
        monitor="val_loss",
        factor=0.5,
        patience=4,
        min_lr=1e-6,
        verbose=1
    ),

    ModelCheckpoint(
        filepath=best_model_path,
        monitor="val_loss",
        save_best_only=True,
        verbose=1
    )
]


# ============================================================
# 18. GPU TRAINING
# ============================================================

EPOCHS = 50

print("\n============================================")
print("STARTING GPU TRAINING")
print("============================================")

print("Batch size:", BATCH_SIZE)
print("Lookback:", LOOKBACK)
print("Epochs:", EPOCHS)

with tf.device("/GPU:0"):

    history = model.fit(
        train_ds,
        validation_data=val_ds,
        epochs=EPOCHS,
        callbacks=callbacks,
        verbose=1
    )


# ============================================================
# 19. SAVE FINAL MODEL
# ============================================================

final_model_path = os.path.join(
    SAVE_DIR,
    "model01.keras"
)

model.save(final_model_path)

print("\nFinal model saved:")
print(final_model_path)


# ============================================================
# 20. LOAD BEST MODEL
# ============================================================

best_model = keras.models.load_model(
    best_model_path
)

print("\nBest model loaded:")
print(best_model_path)


# ============================================================
# 21. TEST PREDICTION
# ============================================================

print("\n============================================")
print("TEST PREDICTION")
print("============================================")

Y_pred = best_model.predict(
    test_ds,
    verbose=1
)

print("Y_pred shape:", Y_pred.shape)


# ============================================================
# 22. GET y_test FROM TEST DATASET
# ============================================================

y_test_batches = []

for _, y_batch in test_ds:

    y_test_batches.append(
        y_batch.numpy()
    )

y_test = np.concatenate(
    y_test_batches,
    axis=0
)

print("y_test shape:", y_test.shape)


# ============================================================
# 23. OVERALL TEST EVALUATION
# ============================================================

mae = mean_absolute_error(
    y_test,
    Y_pred
)

mse = mean_squared_error(
    y_test,
    Y_pred
)

rmse = np.sqrt(mse)

r2 = r2_score(
    y_test,
    Y_pred
)

ev = explained_variance_score(
    y_test,
    Y_pred
)

print("\n============================================")
print("OVERALL TEST RESULTS")
print("============================================")

print(f"MAE                : {mae:.8f}")
print(f"MSE                : {mse:.8f}")
print(f"RMSE               : {rmse:.8f}")
print(f"R²                 : {r2:.8f}")
print(f"Explained Variance : {ev:.8f}")


# ============================================================
# 24. PER-CHANNEL EVALUATION
# ============================================================

results = []

print("\n============================================")
print("PER-CHANNEL TEST RESULTS")
print("============================================")

for i, feature in enumerate(features):

    actual = y_test[:, i]
    pred = Y_pred[:, i]

    mae_i = mean_absolute_error(
        actual,
        pred
    )

    mse_i = mean_squared_error(
        actual,
        pred
    )

    rmse_i = np.sqrt(mse_i)

    std_i = np.std(actual)

    # R2 is not meaningful for constant targets
    if std_i < 1e-8:

        r2_i = np.nan

        ev_i = np.nan

    else:

        r2_i = r2_score(
            actual,
            pred
        )

        ev_i = explained_variance_score(
            actual,
            pred
        )

    print(
        f"{feature:20s} "
        f"MAE={mae_i:.6f} "
        f"RMSE={rmse_i:.6f} "
        f"R²={r2_i}"
    )

    results.append({
        "Energy Channel": feature,
        "MAE": mae_i,
        "MSE": mse_i,
        "RMSE": rmse_i,
        "R2": r2_i,
        "Explained Variance": ev_i,
        "Actual STD": std_i
    })


# ============================================================
# 25. SAVE METRICS
# ============================================================

results_df = pd.DataFrame(results)

metrics_path = os.path.join(
    SAVE_DIR,
    "test_metrics.csv"
)

results_df.to_csv(
    metrics_path,
    index=False
)

print("\nMetrics saved:")
print(metrics_path)


# ============================================================
# 26. SAVE PREDICTIONS
# ============================================================

pred_df = pd.DataFrame(
    Y_pred,
    columns=features
)

prediction_path = os.path.join(
    SAVE_DIR,
    "test_predictions_scaled.csv"
)

pred_df.to_csv(
    prediction_path,
    index=False
)

print("\nPredictions saved:")
print(prediction_path)


# ============================================================
# 27. SAVE TRAINING HISTORY
# ============================================================

history_df = pd.DataFrame(
    history.history
)

history_path = os.path.join(
    SAVE_DIR,
    "training_history.csv"
)

history_df.to_csv(
    history_path,
    index=False
)

print("\nTraining history saved:")
print(history_path)


# ============================================================
# 28. FINAL SAVED FILES
# ============================================================

print("\n============================================")
print("SAVED FILES")
print("============================================")

for filename in sorted(os.listdir(SAVE_DIR)):
    print(
        os.path.join(
            SAVE_DIR,
            filename
        )
    )

TensorFlow: 2.21.0
Available GPUs: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]
Memory growth setting skipped: Physical devices cannot be modified after being initialized
GPU configuration complete.
Save directory: saved_modelv3

Dataset shape: (1736640, 14)
                  time  1.02_1.86_MeV  1.9_2.3_MeV  2.31_3.34_MeV  \
0  2022-09-13 00:00:00       0.035267     0.009745       0.003481   
1  2022-09-13 00:01:00       0.036756     0.012085       0.003330   
2  2022-09-13 00:02:00       0.032474     0.007404       0.003481   
3  2022-09-13 00:03:00       0.033030     0.012080       0.004086   
4  2022-09-13 00:04:00       0.033216     0.013639       0.003329   

   3.4_6.48_MeV  5.84_11_MeV  11.64_23.27_MeV  23.9_32.6_MeV  40.7_68.2_MeV  \
0      0.000167     0.000181         0.000123       0.000021       0.000005   
1      0.000056     0.000000         0.000098       0.000038       0.000001   
2      0.000111     0.000060         0.000160       0.000038       

Model: "sequential_17"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ lstm_19 (LSTM)                  │ (None, 120, 64)        │        19,968 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_19 (Dropout)            │ (None, 120, 64)        │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_20 (LSTM)                  │ (None, 32)             │        12,416 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_20 (Dropout)            │ (None, 32)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_34 (Dense)                │ (None, 32)             │         1,056 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_35 (Dense)                │ (None, 13)             │           429 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 33,869 (132.30 KB)

 Trainable params: 33,869 (132.30 KB)

 Non-trainable params: 0 (0.00 B)


STARTING GPU TRAINING
Batch size: 128
Lookback: 120
Epochs: 50
Epoch 1/50


/home/techpark-2/Desktop/radiation/Shine/sgps-l2-avg1m/.venv/lib/python3.11/site-packages/keras/src/trainers/epoch_iterator.py:74: UserWarning: `shuffle=True` was passed, but will be ignored since the data `x` was provided as a tf.data.Dataset. The Dataset is expected to already be shuffled (via `.shuffle(buffer_size)`).
  self.data_adapter = data_adapters.get_data_adapter(


11528/11532 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - loss: 0.0405 - mae: 0.1251
Epoch 1: val_loss improved from None to 0.03390, saving model to saved_modelv3/best_lstm_model.keras

Epoch 1: finished saving model to saved_modelv3/best_lstm_model.keras
11532/11532 ━━━━━━━━━━━━━━━━━━━━ 85s 7ms/step - loss: 0.0405 - mae: 0.1251 - val_loss: 0.0339 - val_mae: 0.1130 - learning_rate: 0.0010
Epoch 2/50
11530/11532 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - loss: 0.0391 - mae: 0.1189
Epoch 2: val_loss improved from 0.03390 to 0.03383, saving model to saved_modelv3/best_lstm_model.keras

Epoch 2: finished saving model to saved_modelv3/best_lstm_model.keras
11532/11532 ━━━━━━━━━━━━━━━━━━━━ 84s 7ms/step - loss: 0.0391 - mae: 0.1189 - val_loss: 0.0338 - val_mae: 0.1109 - learning_rate: 0.0010
Epoch 3/50
11526/11532 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - loss: 0.0389 - mae: 0.1179
Epoch 3: val_loss improved from 0.03383 to 0.03370, saving model to saved_modelv3/best_lstm_model.keras

Epoch 3: finished saving model t